# 06 - Transfer, the CNN baseline, and the lack-of-fit indicator

The last three results, and the ones the thesis claim actually rests on.

- **the baseline** (§8.3). A CNN that regresses `(xc, yc, R)` straight from the receiver ring.
  It is 100x cheaper than the inversion and it is the honest comparison: if it matches the
  four-stage pipeline, the differentiable forward model bought nothing.
- **out-of-family transfer** (§8.5, §11.2 step 13). Ellipses in two eccentricity tiers, and
  two-void geometries, solved with the FDTD and inverted with a surrogate trained on circles
  only. Nothing is retrained. Three axes are measured separately -- forward prediction,
  sensitivities against the reference solver, and inversion success -- because one number
  covering all three cannot say which of them failed.
- **the lack-of-fit indicator** (§11.2 step 12). The residual misfit as a one-sided test
  statistic for "did the circle family fit this data?", with the threshold *frozen on
  calibration data* before any out-of-family case is scored. It is not a
  "which shape is it" detector and it is not evidence that a flagged case is an ellipse;
  it says the residual is larger than a correctly-specified circle's residual has any
  business being.

The out-of-family data comes from fresh solver runs, not from the network. That is the whole
point: the network has to be asked about a shape it has never seen, and the answer has to be
compared against the real physics of that shape.

**Runtime.** The FDTD solves are minutes; the CNN trains in minutes; the inversions dominate.
The solver's opinion of the transfer adds `1 + 2 * 5` solves per step size, plus two more for
its verdict on the recovered ellipse. Set `QUICK = True` for a short pass. Headless:

```
modal run modal_app.py::transfer_and_detector
```

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

In [ ]:
QUICK = True

N_IN = 16 if QUICK else 32            # in-family cases, split calibration / held-out test
N_HARD = 6 if QUICK else 12           # in-family but *difficult*: the false-positive controls
N_ELLIPSE = 6 if QUICK else 12        # out-of-family: ellipses, half mild + half wide
N_TWO = 6 if QUICK else 12            # out-of-family: two voids
N_SEED = 6 if QUICK else 12           # stage-0 seeding comparison
REG_EPOCHS = 30 if QUICK else 200
SNR = 30.0                            # the SNR every number in this notebook is quoted at
SNR_HARD = 20.0                       # the controls' SNR: correctly specified, just noisier
AR_MILD = (1.05, 1.30)                # eccentricity tiers: mildly out of family...
AR_WIDE = (1.50, 2.40)                # ...and far enough out to be another shape
SENS_H_F = (0.02,)                    # out-of-family 9b: 1 + 2*5*len(SENS_H_F) FDTD solves

print(f"{'QUICK' if QUICK else 'FULL'}: {N_IN} in-family ({N_IN // 2} calibration + "
      f"{N_IN - N_IN // 2} held out) + {N_HARD} hard controls, "
      f"{N_ELLIPSE} ellipse + {N_TWO} two-void out-of-family, at {SNR:.0f} dB")
print(f"ellipse tiers: aspect {AR_MILD[0]}-{AR_MILD[1]} ({N_ELLIPSE // 2} cases) and "
      f"{AR_WIDE[0]}-{AR_WIDE[1]} ({N_ELLIPSE - N_ELLIPSE // 2} cases)")
print(f"RingCNN: {REG_EPOCHS} epochs")

In [ ]:
import csv

import h5py

from src import training
from src.data import generate as G
from src.data.dataset import load_incident, load_inversion_case
from src.geometry.sdf import (Circle, Ellipse, TwoCircle, equivalent_circle,
                              fine_coords, geometry_channels, material_fields,
                              soft_indicator)
from src.inverse import invert as INV
from src.inverse import sensitivity as SENS
from src.inverse.misfit import InverseCase, Objective, SurrogateForward
from src.models import cnn_regressor as CNN
from src.solver import harmonic as H
from src.solver.fdtd_elastic import ElasticFDTD2D

for k in ("train", "val", "test"):
    assert paths[k].exists(), f"missing {paths[k]} -- run notebook 02"
assert CKPT.exists(), "no checkpoint -- run notebook 03"

model, meta = training.load(CKPT, device=DEV)
inc = load_incident(str(paths["test"]), device=DEV)
fwd = SurrogateForward(model, inc, device=DEV)
circle, ellipse, twocircle = Circle(), Ellipse(), TwoCircle()
om = H.omegas_tensor(DEV)

with h5py.File(paths["test"], "r") as f:
    src_test = f["samples/src_idx"][:]
    nu_test = f["samples/nu_idx"][:]
    theta_test = f["samples/theta"][:]
print(f"model {meta['arch']}, epoch {meta['epoch']}, device {DEV}")

## Part 1 -- the baseline that has to be beaten

A circular 1-D CNN over the 32 receivers, `4M + 3 = 83` input channels: real and imaginary
parts of both displacement components at all 20 frequencies, plus three broadcast conditioning
channels (source position and centred Poisson ratio). It regresses the **unconstrained** `z`,
not `theta`, because an MSE on `theta` would weight the two positions about 40x more heavily
than the radius purely through their units and the radius would never be learned.

The input is normalised by the **receiver-space** incident scale, not the domain scale. Those
differ by roughly two orders of magnitude -- the domain scale is set by the near-source
singularity -- and using the wrong one compresses the whole input to the bottom of float32's
useful range. Notebook 02 measured that ratio; this is the second place it matters.

Noise is added at the same `SNR` and in the same order (on the total A-scans, before the
incident field is subtracted) as the inversion sees, so the two are compared on the same data
and not on two different problems. It is also scored by the same code: `CNN.score` wraps each
prediction in the same result object the inversion returns and calls the inversion's own
`summarise`, so "position error" and "success" mean one thing in this notebook rather than two.

In [ ]:
gtr = torch.Generator().manual_seed(cfg.SEED)
gva = torch.Generator().manual_seed(cfg.SEED + 1)
gte = torch.Generator().manual_seed(cfg.SEED + 2)

t0 = time.perf_counter()
rtr = CNN.ring_features(str(paths["train"]), snr_db=SNR, generator=gtr, device=DEV)
rva = CNN.ring_features(str(paths["val"]), snr_db=SNR, generator=gva, device=DEV)
rte = CNN.ring_features(str(paths["test"]), snr_db=SNR, generator=gte, device=DEV)
print(f"ring features in {time.perf_counter()-t0:.1f} s")
print(f"train {tuple(rtr.x.shape)}  val {tuple(rva.x.shape)}  test {tuple(rte.x.shape)}")
print(f"RING_CHANNELS = {CNN.RING_CHANNELS} = 4 x {cfg.M_FREQ} + 3   "
      f"({rtr.x.numel()*4/1e6:.1f} MB in memory for train)")

In [ ]:
net, hist = CNN.train_regressor(rtr, rva, device=DEV, epochs=REG_EPOCHS,
                                log_every=max(REG_EPOCHS // 8, 1))
CNN.save(net, E.checkpoints / "ringcnn.pt")
print(f"\n{net.n_params():,} parameters "
      f"({net.n_params()/model.effective_params():.3%} of the FNO's)")

held_te = np.isin(src_test, cfg.SRC_HELDOUT)
sc_val = CNN.score(net, rva)
sc_te = CNN.score(net, rte)
sub = lambda m: CNN.RingData(rte.x[m], rte.theta[m], rte.nu[m], rte.src_idx[m])
m_he = torch.from_numpy(held_te).to(rte.x.device)
sc_tr_src = CNN.score(net, sub(~m_he))
sc_he_src = CNN.score(net, sub(m_he))

table([(k, f"{sc_val[k]:.4f}", f"{sc_te[k]:.4f}", f"{sc_tr_src[k]:.4f}",
        f"{sc_he_src[k]:.4f}")
       for k in ("position_ls_mean", "position_ls_median", "position_ls_p90",
                 "radius_ls_median", "iou_median", "success_rate")],
      ["RingCNN", "val", "test", "test: trained src", f"test: held out {cfg.SRC_HELDOUT}"])
print(f"success = position < {cfg.GATE_POSITION_LS} lambda_s *and* IoU > {cfg.GATE_IOU}; "
      "same definition, same function, as the inversion's success rate")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].semilogy(hist["train"], lw=1.0, label="train")
ax[0].semilogy(hist["val"], lw=1.0, label="val")
ax[0].set(xlabel="epoch", ylabel="MSE on z", title="RingCNN training")
ax[0].legend(fontsize=8)

with torch.no_grad():
    th_hat = net.predict_theta(rte.x, rte.nu, circle)
lam_te = np.array([cfg.cs_over_cp(float(v)) / cfg.FC for v in _np(rte.nu)])
err_cnn = (_np((th_hat[:, :2] - rte.theta[:, :2]).pow(2).sum(-1).sqrt()) / lam_te)
bins = np.linspace(0, max(err_cnn.max() * 1.02, 0.5), 40)
ax[1].hist(err_cnn[~held_te], bins=bins, alpha=0.75, label="trained sources")
ax[1].hist(err_cnn[held_te], bins=bins, alpha=0.75, label=f"held out {cfg.SRC_HELDOUT}")
ax[1].axvline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0,
              label=f"gate {cfg.GATE_POSITION_LS} lambda_s")
ax[1].set(xlabel="position error / lambda_s", ylabel="count",
          title="RingCNN on the test split")
ax[1].legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_ringcnn.png")
plt.show()

### Does stage 0 actually help?

`invert` *adds* the CNN's guess to the screen's survivors rather than replacing them, and the
asymmetry is deliberate: if the CNN is right the extra candidate costs one row in a batch of 17,
and if the defect is out of distribution -- which is what the rest of this notebook is about --
the CNN's guess can be badly wrong and must not be the only starting point.

The comparison below is the same cases inverted with and without the seed. What is being looked
for is not a better final answer -- both should reach the same basin on in-family data -- but a
cheaper path to it, visible as a lower stage-2 objective from the first step.

In [ ]:
rng = np.random.default_rng(cfg.SEED + 7)
sel_seed = rng.choice(len(theta_test), size=N_SEED, replace=False)


def get_case(i, snr_db=SNR, seed_off=0):
    g = torch.Generator().manual_seed(int(cfg.SEED) + int(i) + int(seed_off))
    return InverseCase.from_dict(
        load_inversion_case(str(paths["test"]), int(i), snr_db=snr_db,
                            generator=g)).to(DEV)


rows, seed_rec = [], []
for i in sel_seed:
    case = get_case(int(i))
    j = int(nu_test[i])
    x = CNN.pack_ring(case.d_obs.cpu(), src_idx=torch.tensor([case.src_idx]),
                      nu=torch.tensor([case.nu]),
                      scale=inc["scale_recv"][case.src_idx, j].cpu().unsqueeze(0))
    with torch.no_grad():
        th0 = net.predict_theta(x.to(DEV), torch.tensor([case.nu], device=DEV),
                                circle)[0]
    r_no = INV.invert(fwd, case, family=circle)
    r_yes = INV.invert(fwd, case, family=circle, theta_init=th0)
    e0 = float((th0[:2].cpu() - case.theta_true[:2].cpu()).norm()) / case.lambda_s
    rows.append((int(i), f"{e0:.3f}", f"{r_no.position_error_ls:.4f}",
                 f"{r_yes.position_error_ls:.4f}",
                 f"{r_no.stages['stage2_trace'][0]:.3e}",
                 f"{r_yes.stages['stage2_trace'][0]:.3e}"))
    seed_rec.append(dict(index=int(i), stage0_error_ls=e0,
                         no_seed=r_no.position_error_ls,
                         seeded=r_yes.position_error_ls,
                         no_seed_misfit=r_no.misfit, seeded_misfit=r_yes.misfit))
table(rows, ["test i", "stage 0 err", "final, no seed", "final, seeded",
             "stage 2 J[0], no seed", "stage 2 J[0], seeded"])
d0 = np.array([r["no_seed"] for r in seed_rec])
d1 = np.array([r["seeded"] for r in seed_rec])
print(f"\nmedian position error  no seed {np.median(d0):.4f}   "
      f"seeded {np.median(d1):.4f} lambda_s")

## Part 2 -- out-of-family geometries, solved properly

Ellipses in two eccentricity tiers -- aspect ratio 1.05-1.30 and 1.5-2.4, random orientation --
and pairs of circles (separation 1.6-3.0 mean radii). Both families are in `sdf.py` and neither
appears anywhere in the training data.

The tiers are what makes this a graded experiment rather than a verdict. A single hard tier can
only say "it worked" or "it did not", and if it did not, eccentricity, the input distribution and
the optimiser are all still live candidates. A mild tier is close enough to a circle that the
`Ellipse` family reduces to `Circle` in the limit -- `sdf.py` normalises `s - 1` rather than
`s^2 - 1` so that `a = b = R` returns `r - R` exactly, not 17% short at `1.5 R` -- so a mild-tier
failure cannot be blamed on eccentricity, and a mild pass with a wide failure locates where the
transfer stops. The interface width is held at the trained value for every tier: it is a
separate axis and mixing it in here would measure two things at once.

The solver run is set up exactly as `generate.py` sets it up -- same fine grid, same interface
width in *physical* units (`EPS_LEN_PHYS`, the width the network sees, not that many cells of
the finer grid), same source injection, same receiver sampling from the downsampled field. The
only change is which SDF makes `chi`. Anything else would confound "the network has not seen
this shape" with "the data was made differently".

Sources are drawn from `SRC_TRAIN` on purpose. The point of this experiment is to vary one
thing, and that thing is the shape.

**The equivalent circle.** Scoring a circle fit against a non-circular truth needs a reference.
The convention here is equal area: `R_eq = sqrt(a b)` for an ellipse, `sqrt(R1^2 + R2^2)` for
two voids, with the area-weighted centroid as the centre. It is a convention and not a ground
truth -- there is no correct circle for an ellipse -- which is exactly why the residual misfit,
and not the position error, is what the lack-of-fit indicator is built on.

In [ ]:
def sample_out(n, fam, rng, ar_range=(1.5, 2.4)):
    th, ss, jj = [], [], []
    while len(th) < n:
        j = int(rng.integers(len(cfg.NU_LIST)))
        lam = cfg.cs_over_cp(cfg.NU_LIST[j]) / cfg.FC
        s = int(rng.choice(cfg.SRC_TRAIN))
        sx, sy = cfg.SOURCE_XY[s]
        if fam.name == "ellipse":
            r_eq = float(rng.uniform(1.15 * cfg.R_MIN_LS, 0.85 * cfg.R_MAX_LS)) * lam
            ar = float(rng.uniform(*ar_range))
            a, b = r_eq * math.sqrt(ar), r_eq / math.sqrt(ar)
            ext = a
        else:
            r1 = float(rng.uniform(cfg.R_MIN_LS, 0.8 * cfg.R_MAX_LS)) * lam
            r2 = r1 * float(rng.uniform(0.6, 1.0))
            # 1.6-3.0 mean radii apart: overlapping peanut at the low end,
            # two resolved voids at the high end.
            sep = 0.5 * (r1 + r2) * float(rng.uniform(1.6, 3.0))
            ang = float(rng.uniform(0, 2 * math.pi))
        pad = cfg.BOUNDARY_KEEPOUT_LS * lam
        if fam.name == "ellipse":
            keep = ext + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            if math.hypot(xc - sx, yc - sy) < ext + G.SRC_KEEPOUT_LS * lam:
                continue
            al = float(rng.uniform(-math.pi / 2, math.pi / 2))
            th.append([xc, yc, a, b, al])
        else:
            keep = max(r1, r2) + 0.5 * sep + pad
            if 2 * keep >= cfg.L_DOMAIN:
                continue
            xc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            yc = float(rng.uniform(keep, cfg.L_DOMAIN - keep))
            dx_, dy_ = 0.5 * sep * math.cos(ang), 0.5 * sep * math.sin(ang)
            c1 = (xc + dx_, yc + dy_)
            c2 = (xc - dx_, yc - dy_)
            ok = all(math.hypot(c[0] - sx, c[1] - sy) > r + G.SRC_KEEPOUT_LS * lam
                     for c, r in ((c1, r1), (c2, r2)))
            ok &= all(pad + r <= v <= cfg.L_DOMAIN - pad - r
                      for c, r in ((c1, r1), (c2, r2)) for v in c)
            if not ok:
                continue
            th.append([c1[0], c1[1], r1, c2[0], c2[1], r2])
        ss.append(s)
        jj.append(j)
    return (np.asarray(th, np.float32), np.asarray(ss, np.int64),
            np.asarray(jj, np.int64))


def eq_circle(theta, fam):
    """numpy wrapper on geometry.sdf.equivalent_circle, for tables and records."""
    t = torch.as_tensor(np.atleast_2d(np.asarray(theta, np.float64)))
    return equivalent_circle(t, fam).numpy()


rng = np.random.default_rng(cfg.SEED + 11)
n_mild = N_ELLIPSE // 2
th_m, src_m, nu_m = sample_out(n_mild, ellipse, rng, AR_MILD)
th_w, src_w, nu_w = sample_out(N_ELLIPSE - n_mild, ellipse, rng, AR_WIDE)
th_e = np.concatenate([th_m, th_w])
src_e = np.concatenate([src_m, src_w])
nu_e = np.concatenate([nu_m, nu_w])
mild = np.arange(len(th_e)) < n_mild          # the tier mask every ellipse table splits on
th_t, src_t, nu_t = sample_out(N_TWO, twocircle, rng)
print(f"ellipses  {th_e.shape}  aspect ratios "
      f"{np.round(th_e[:, 2]/th_e[:, 3], 2)}  (the first {n_mild} are the mild tier)")
print(f"two-void  {th_t.shape}  radius ratios "
      f"{np.round(th_t[:, 5]/th_t[:, 2], 2)}")

In [ ]:
inc_ascans = inc["ascans"].cpu()
inc_scale_r = inc["scale_recv"].cpu()


def solve_out(theta, fam, src_idx, nu_idx, *, snr_db=SNR, seed=0):
    """FDTD -> scattered displacement phasors at the ring, [B,R,2,M] complex."""
    yy_f, xx_f = fine_coords(device=DEV)
    out = []
    for lo in range(0, len(theta), cfg.GEN_BATCH):
        hi = min(lo + cfg.GEN_BATCH, len(theta))
        th = torch.as_tensor(theta[lo:hi], device=DEV)
        chi = soft_indicator(fam.sdf(th, yy_f, xx_f), G.EPS_LEN_PHYS)
        lm = [cfg.lame_from_nu(cfg.NU_LIST[int(j)]) for j in nu_idx[lo:hi]]
        lam0 = torch.tensor([v[0] for v in lm], device=DEV).view(-1, 1, 1)
        mu0 = torch.tensor([v[1] for v in lm], device=DEV).view(-1, 1, 1)
        lam, mu, rho = material_fields(chi, lam0, mu0)
        sim = ElasticFDTD2D(lam, mu, rho)
        src = [cfg.net_to_fine(*cfg.SOURCES_NET[int(s)]) for s in src_idx[lo:hi]]
        res = sim.run(src, nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
        a_tot = res.ascans.cpu()
        a_inc = inc_ascans[torch.as_tensor(src_idx[lo:hi]),
                           torch.as_tensor(nu_idx[lo:hi])]
        if snr_db is not None:
            g = torch.Generator().manual_seed(int(cfg.SEED) + seed + lo)
            a_tot = H.add_measurement_noise(a_tot, snr_db, generator=g)
        d = (H.displacement_from_ascans(a_tot, omegas=om)
             - H.displacement_from_ascans(a_inc, omegas=om))
        out.append(d)
        del sim, chi, lam, mu, rho
    if DEV.startswith("cuda"):
        torch.cuda.empty_cache()
    return torch.cat(out)


t0 = time.perf_counter()
d_e = solve_out(th_e, ellipse, src_e, nu_e, seed=100)
d_t = solve_out(th_t, twocircle, src_t, nu_t, seed=200)
print(f"{len(th_e)+len(th_t)} out-of-family FDTD solves in "
      f"{(time.perf_counter()-t0)/60:.1f} min")
print(f"d_ellipse {tuple(d_e.shape)}   d_two {tuple(d_t.shape)}")

# The case carries the *real* out-of-family truth and the family it is written in, not a
# pre-reduced equal-area circle.  That is what lets InversionResult score IoU against the
# actual ellipse -- the number that says a circle cannot represent it -- while still
# reporting a position error, via the equal-area reduction, inside the library where the
# reduction is documented.  eq_e / eq_t are kept only for the tables and the record.
eq_e = eq_circle(th_e, ellipse)
eq_t = eq_circle(th_t, twocircle)
cases_e = [InverseCase(d_obs=d_e[k:k+1], src_idx=int(src_e[k]),
                       nu_idx=int(nu_e[k]), snr_db=SNR,
                       theta_true=torch.tensor(th_e[k], dtype=torch.float32),
                       truth_family=ellipse).to(DEV) for k in range(len(th_e))]
cases_t = [InverseCase(d_obs=d_t[k:k+1], src_idx=int(src_t[k]),
                       nu_idx=int(nu_t[k]), snr_db=SNR,
                       theta_true=torch.tensor(th_t[k], dtype=torch.float32),
                       truth_family=twocircle).to(DEV) for k in range(len(th_t))]

In [ ]:
# Axis 1 of three: forward prediction, at the *true* out-of-family geometry.  No
# inversion and no extra solves are involved -- the surrogate is handed a shape it was
# never trained on and asked for the receiver phasors, and the FDTD answer is already in
# hand from the cell above.  Keeping this separate from inversion success is the point:
# a good field prediction with a failed inversion indicts the optimiser, the reverse
# indicts the network, and one number for both cannot tell them apart.
def _rel(a, b):
    return float((a - b).abs().pow(2).sum().sqrt()
                 / b.abs().pow(2).sum().sqrt().clamp_min(1e-30))


def fwd_err(theta, fam, src_idx, nu_idx, d_ref):
    return np.array([_rel(fwd.predict(torch.as_tensor(theta[k:k+1], device=DEV), fam,
                                      src_idx=int(src_idx[k]), nu_idx=int(nu_idx[k])),
                          d_ref[k:k+1, ..., cfg.BAND_STAGE3].to(DEV))
                     for k in range(len(theta))])


fw_e = fwd_err(th_e, ellipse, src_e, nu_e, d_e)
fw_t = fwd_err(th_t, twocircle, src_t, nu_t, d_t)
table([("ellipse, mild tier", f"{np.mean(th_e[mild, 2]/th_e[mild, 3]):.2f}",
        f"{np.median(fw_e[mild]):.4f}", f"{fw_e[mild].max():.4f}"),
       ("ellipse, wide tier", f"{np.mean(th_e[~mild, 2]/th_e[~mild, 3]):.2f}",
        f"{np.median(fw_e[~mild]):.4f}", f"{fw_e[~mild].max():.4f}"),
       ("two voids", "-", f"{np.median(fw_t):.4f}", f"{fw_t.max():.4f}")],
      ["out-of-family set", "mean a/b", "median rel err", "worst"])
print(f"\nreceiver-space relative error at the true geometry, band "
      f"{cfg.BAND_STAGE3.start}-{cfg.BAND_STAGE3.stop}, {SNR:.0f} dB data.")
print(f"the in-family reference for this quantity is stage D's; the field gate there is "
      f"{cfg.GATE_REL_L2}.")

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(11.6, 5.8))
for r_, (th_, fam, eq, ttl) in enumerate([(th_e, ellipse, eq_e, "ellipse"),
                                          (th_t, twocircle, eq_t, "two voids")]):
    for c_ in range(3):
        k = c_ % len(th_)
        _, chi = geometry_channels(torch.as_tensor(th_[k:k+1], device=DEV), fam)
        _, chi_eq = geometry_channels(torch.as_tensor(eq[k:k+1], dtype=torch.float32,
                                                      device=DEV), circle)
        gx = np.arange(cfg.N_NET) * cfg.DX_NET
        a_ = ax[r_, c_]
        a_.imshow(_np(chi)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
                  extent=[gx[0], gx[-1], gx[0], gx[-1]])
        a_.contour(gx, gx, _np(chi_eq)[0], levels=[0.5], colors="C3",
                   linewidths=1.2)
        sx, sy = cfg.SOURCE_XY[int((src_e if r_ == 0 else src_t)[k])]
        a_.plot(sx, sy, "C0*", ms=10)
        a_.set(title=f"{ttl} {k}", xticks=[], yticks=[])
        a_.grid(False)
    a_ = ax[r_, 3]
    dd = (d_e if r_ == 0 else d_t)[0]
    im = a_.imshow(_np(dd.abs()[:, 0, :]), origin="lower", aspect="auto",
                   extent=[cfg.FREQS[0], cfg.FREQS[-1], 0, cfg.N_RECV],
                   cmap="magma")
    a_.set(xlabel="f / f_c", ylabel="receiver",
           title=f"|d_obs| x-component\n{ttl} 0, {SNR:.0f} dB")
    a_.grid(False)
    fig.colorbar(im, ax=a_, fraction=0.046)
ax[0, 0].set_ylabel("red: equal-area circle")
fig.tight_layout()
savefig(fig, "06_out_of_family_shapes.png")
plt.show()

## Part 3 -- the lack-of-fit indicator

Every case is inverted with `Circle()`, and the question asked afterwards is only ever *"could
a circle have produced this data?"* -- never *"what shape is it?"*. Three separate claims got
retired to make that the question:

- a large residual is not evidence about **shape**. It is equally consistent with a wrong
  source, a wrong material, unmodelled noise, or an optimiser that stopped early. Attributing
  it needs the reference solver -- `inverse.sensitivity.verify_with_solver`, which Part 4 runs
  on the transferred answer -- and not this statistic.
- a small residual does not certify correctness. The optimiser is free to exploit surrogate
  error to reach a low misfit at a wrong geometry.
- the residual is not a gift of the physics penalty. Any supervised forward surrogate has one.

**The statistic.** `INV.lack_of_fit_statistic` normalises the converged data residual by what
noise and model error already explain,

    T = J_data / (eta * 10^(-SNR/10) + LOF_MODEL_FLOOR)

so `T ~ 1` means "as well fitted as it is possible to fit" and `T >> 1` means "the residual is
larger than a correctly-specified circle's residual has any business being". Three details
carry weight. `J_data` **excludes** the Tikhonov term -- `InversionResult.misfit` does,
`stages["stage3_trace"]` does not -- because a regulariser's contribution varies with how close
the solution sat to the bounds, which is noise in exactly the quantity being thresholded. The
`LOF_MODEL_FLOOR` = `GATE_REL_L2^2` = 0.0025 is a **nonzero** model-error floor; without it
every high-SNR case reads as catastrophic lack of fit, because it is being compared against a
noise level far below the surrogate's own accuracy. And `eta` is the deconvolution's noise
amplification (`H.conditioning_report`), because flat time-domain noise does not stay flat
after dividing by `i omega s_hat` -- a single scalar for it is still an approximation, since
the finite window also correlates the noise across frequency.

**The threshold is frozen before the test.** The in-family cases are split in half: one half
*calibrates* the threshold as the `1 - GATE_LOF_FPR` quantile of its own statistics, one-sided,
without ever looking at an ellipse; the other half is held out and is what the reported
false-positive rate is measured on. Choosing the threshold by Youden's index on the test data
-- which is what this notebook did in v2.0 -- reports the best operating point available in
hindsight and is not a rate anything can be expected to reproduce.

**The false-positive controls are the hard part.** Calibrating on easy in-family cases sets the
threshold too low, and then every genuinely difficult but *correctly specified* case is flagged
as a mismatch. So `N_HARD` of the in-family cases are the smallest radii in the pool at
`SNR_HARD` = 20 dB rather than 30 dB -- correctly specified, just hard -- and they are split
between calibration and held-out test alongside the easy ones.

In [ ]:
# Three pools, all inverted as circles.  In-family cases come from the test split at the
# same SNR and the same source pool, so the null class differs from the alternative in
# the shape and in nothing else.
pool = np.where(np.isin(src_test, cfg.SRC_TRAIN))[0]
lam_test = np.array([cfg.cs_over_cp(cfg.NU_LIST[int(j)]) / cfg.FC for j in nu_test])
r_ls = theta_test[:, 2] / lam_test                  # radius in shear wavelengths

sel_in = rng.choice(pool, size=N_IN, replace=False)
rest = np.setdiff1d(pool, sel_in)
sel_hard = rest[np.argsort(r_ls[rest])][:N_HARD]    # the smallest radii in the pool

n_c, n_ch = N_IN // 2, N_HARD // 2
easy = [get_case(int(i), seed_off=500) for i in sel_in]
hard = [get_case(int(i), snr_db=SNR_HARD, seed_off=700) for i in sel_hard]
cases_cal = easy[:n_c] + hard[:n_ch]                # sets the threshold
cases_ho = easy[n_c:] + hard[n_ch:]                 # measures the false-positive rate
print(f"in family: {len(cases_cal)} calibration + {len(cases_ho)} held out, of which "
      f"{N_HARD} hard controls at {SNR_HARD:.0f} dB with radii "
      f"{r_ls[sel_hard].min():.2f}-{r_ls[sel_hard].max():.2f} l_s "
      f"(easy: {r_ls[sel_in].min():.2f}-{r_ls[sel_in].max():.2f} l_s)")

t0 = time.perf_counter()
res_cal = INV.run_many(fwd, cases_cal, family=circle, progress=tqdm)
res_ho = INV.run_many(fwd, cases_ho, family=circle, progress=tqdm)
res_e = INV.run_many(fwd, cases_e, family=circle, progress=tqdm)
res_t = INV.run_many(fwd, cases_t, family=circle, progress=tqdm)
print(f"\n{len(res_cal)+len(res_ho)+len(res_e)+len(res_t)} inversions in "
      f"{(time.perf_counter()-t0)/60:.1f} min")

In [ ]:
# Misfit -> statistic, at each case's own SNR.  `amp` is hoisted out of the loop only
# because lack_of_fit_statistic would otherwise recompute the conditioning report per
# case; the value is identical.
amp = float(torch.tensor(H.conditioning_report()["amplification"]).pow(2).mean())

def lof_stats(res, cases):
    return [INV.lack_of_fit_statistic(r.misfit, snr_db=c.snr_db, amplification=amp)
            for r, c in zip(res, cases)]

T_cal = lof_stats(res_cal, cases_cal)
T_ho = lof_stats(res_ho, cases_ho)
T_e = lof_stats(res_e, cases_e)
T_t = lof_stats(res_t, cases_t)

# Frozen here, on calibration data only.  stats_out is deliberately not passed: the
# threshold is the (1 - GATE_LOF_FPR) quantile of the in-family statistics and must not
# be tuned against the out-of-family set.
lof = INV.calibrate_lack_of_fit(
    T_cal, target_fpr=cfg.GATE_LOF_FPR, snr_db=(SNR, SNR_HARD),
    note=f"nb06 QUICK={QUICK}: {n_c} easy + {n_ch} hard circles from the test split")
ev = lof.evaluate(T_ho, T_e + T_t)
ev_e = lof.evaluate(T_ho, T_e)
ev_t = lof.evaluate(T_ho, T_t)

# The ROC is a diagnostic on the *held-out* half only -- an AUC near 0.5 would say no
# threshold can work -- and never the operating point.
roc = INV.lack_of_fit_roc(T_ho, T_e + T_t)
roc_e = INV.lack_of_fit_roc(T_ho, T_e)
roc_t = INV.lack_of_fit_roc(T_ho, T_t)

res_in, mis_in = res_cal + res_ho, [r.misfit for r in res_cal + res_ho]
mis_e, mis_t = [r.misfit for r in res_e], [r.misfit for r in res_t]
T_in = T_cal + T_ho

table([("circle, calibration", len(T_cal), f"{np.median(T_cal):.3g}",
        f"{INV.summarise(res_cal)['position_ls_median']:.4f}",
        f"{np.mean([t > lof.threshold for t in T_cal]):.2f}", "-"),
       ("circle, held out", len(T_ho), f"{np.median(T_ho):.3g}",
        f"{INV.summarise(res_ho)['position_ls_median']:.4f}",
        f"{ev['fpr']:.2f}", "-"),
       ("ellipse", len(T_e), f"{np.median(T_e):.3g}",
        f"{INV.summarise(res_e)['position_ls_median']:.4f}",
        f"{ev_e['tpr']:.2f}", f"{roc_e['auc']:.3f}"),
       ("two voids", len(T_t), f"{np.median(T_t):.3g}",
        f"{INV.summarise(res_t)['position_ls_median']:.4f}",
        f"{ev_t['tpr']:.2f}", f"{roc_t['auc']:.3f}")],
      ["inverted as a circle", "n", "median T", "median position err (l_s)",
       "flagged", "AUC vs held-out"])
print(f"\nthreshold T > {lof.threshold:.3g}, frozen at a target FPR of "
      f"{lof.target_fpr:.2f} on {lof.n_in} calibration cases "
      f"(achieved {lof.calibration_fpr:.2f} there)")
print(f"held out: FPR {ev['fpr']:.2f} vs gate {ev['gate']:.2f} -> "
      f"{'PASS' if ev['gate_pass'] else 'FAIL'};  TPR {ev['tpr']:.2f};  "
      f"AUC {roc['auc']:.4f}")
print(f"median T = {roc['median_in']:.3g} (in family) vs {roc['median_out']:.3g} "
      f"(out), a factor of {roc['median_out']/max(roc['median_in'],1e-30):.1f}")
if np.median(T_in) > 3.0:
    print(f"\nNOTE: in-family median T = {np.median(T_in):.1f} >> 1, so the residual is "
          f"dominated by surrogate error rather than by noise and LOF_MODEL_FLOOR = "
          f"{cfg.LOF_MODEL_FLOOR:.2e} understates it.  The threshold still separates the "
          "classes; what it is measuring is the surrogate, not the measurement.")
elif np.median(T_in) < 0.3:
    print(f"\nNOTE: in-family median T = {np.median(T_in):.3f} << 1.  Nothing fits better "
          "than noise permits; the denominator is overstated, because eta averages the "
          "deconvolution amplification over the band while the misfit is dominated by the "
          "well-conditioned frequencies.  T is still monotone in the residual and the "
          "threshold is still frozen, so the FPR is valid -- but T = 1 should not be read "
          "as 'the best possible fit' at this SNR.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11.6, 3.2))
ax[0].plot(roc["fpr"], roc["tpr"], "-", lw=1.4,
           label=f"all out-of-family, AUC {roc['auc']:.3f}")
ax[0].plot(roc_e["fpr"], roc_e["tpr"], "--", lw=1.0,
           label=f"ellipse only, AUC {roc_e['auc']:.3f}")
ax[0].plot(roc_t["fpr"], roc_t["tpr"], ":", lw=1.2,
           label=f"two voids only, AUC {roc_t['auc']:.3f}")
ax[0].plot([0, 1], [0, 1], c="0.6", lw=0.8)
ax[0].plot(ev["fpr"], ev["tpr"], "k*", ms=11,
           label=f"frozen: TPR {ev['tpr']:.2f} at FPR {ev['fpr']:.2f}")
ax[0].axvline(cfg.GATE_LOF_FPR, ls=":", c="C3", lw=1.0)
ax[0].set(xlabel="false positive rate (in-family flagged)",
          ylabel="true positive rate (out-of-family flagged)",
          title="figure 5: lack-of-fit indicator\n(curve = held-out diagnostic, "
                "star = frozen operating point)", aspect="equal")
ax[0].legend(fontsize=7)

for i, v in enumerate([T_cal, T_ho, T_e, T_t]):
    j = np.full(len(v), i, float) + rng.normal(0, 0.06, len(v))
    ax[1].semilogy(j, v, "o", ms=5, alpha=0.7, c=f"C{0 if i < 2 else i - 1}")
    ax[1].semilogy([i - 0.25, i + 0.25], [np.median(v)] * 2, "k-", lw=1.6)
ax[1].axhline(lof.threshold, ls="--", c="C3", lw=1.0,
              label=f"frozen threshold {lof.threshold:.3g}")
ax[1].axhline(1.0, ls=":", c="0.5", lw=1.0, label="T = 1: noise + model floor")
ax[1].set(xticks=[0, 1, 2, 3],
          xticklabels=["circle\n(calib)", "circle\n(held out)", "ellipse",
                       "two voids"],
          ylabel="lack-of-fit statistic T", title="the statistic itself")
ax[1].legend(fontsize=7.5)

pos_in = [r.position_error_ls for r in res_ho]
pos_out = [r.position_error_ls for r in res_e + res_t]
ax[2].loglog(T_ho, np.maximum(pos_in, 1e-4), "o", ms=5, alpha=0.75,
             label="in family (held out)")
ax[2].loglog(T_e + T_t, np.maximum(pos_out, 1e-4), "s", ms=5, alpha=0.75,
             label="out of family (vs equal-area circle)")
ax[2].axhline(cfg.GATE_POSITION_LS, ls="--", c="C3", lw=1.0)
ax[2].axvline(lof.threshold, ls="--", c="0.4", lw=1.0)
ax[2].set(xlabel="lack-of-fit statistic T", ylabel="position error / lambda_s",
          title="it still localises --\nit just knows it does not fit")
ax[2].legend(fontsize=7)
fig.tight_layout()
savefig(fig, "06_fig5_lack_of_fit_roc.png")
plt.show()

## Part 4 -- the transfer, with no retraining

The surrogate was trained on circles. It is now asked about ellipses, and the *only* thing that
changes is which `ShapeFamily` builds the two geometry input channels. The weights are frozen,
the band is the same, the optimiser is the same.

Stage 1 is skipped, and not for convenience. `screen_candidates` is family-general -- it reads
`fam.param_names` to find the centre slots, so it scans an ellipse's `(xc, yc)` on the same
lattice as a circle's and leaves `(a, b, alpha)` at the midpoint of their bounds -- but a
position scan over an ellipse whose axes are fixed at the bound midpoint is a worse seed than
the converged circle fit already in hand. So the ellipse run is seeded from that fit --
`(xc, yc, R) -> (xc, yc, a=R, b=R, alpha=0)`, a circle expressed in ellipse coordinates -- and
refined from there. This is the honest version of the §8.5 claim: the transfer works because
the network learned an operator on `(phi_tilde, chi)` fields rather than a map on three numbers,
and the evidence is that releasing two extra degrees of freedom *lowers the misfit* on data the
network has never seen the shape of.

In [ ]:
res_tr, rows = [], []
for k, (ce, rc) in enumerate(zip(cases_e, res_e)):
    x0, y0, r0 = (float(v) for v in rc.theta[:3])
    seed = torch.tensor([x0, y0, r0, r0, 0.0], dtype=torch.float32, device=DEV)
    ce5 = InverseCase(d_obs=ce.d_obs, src_idx=ce.src_idx, nu_idx=ce.nu_idx,
                      snr_db=ce.snr_db,
                      theta_true=torch.tensor(th_e[k], dtype=torch.float32)
                      ).to(DEV)
    r = INV.invert(fwd, ce5, family=ellipse, theta_init=seed, skip_screen=True)
    res_tr.append(r)
    tt, th_hat = th_e[k], _np(r.theta)
    ar_true, ar_hat = tt[2] / tt[3], th_hat[2] / max(th_hat[3], 1e-9)
    da = abs(((th_hat[4] - tt[4] + math.pi / 2) % math.pi) - math.pi / 2)
    rows.append((k, f"{ar_true:.2f}", f"{ar_hat:.2f}",
                 f"{math.degrees(da):.1f}", f"{r.position_error_ls:.4f}",
                 f"{rc.misfit:.3e}", f"{r.misfit:.3e}",
                 f"{100*(1 - r.misfit/max(rc.misfit,1e-30)):+.0f}%"))
table(rows, ["case", "true a/b", "fitted a/b", "orientation err (deg)",
             "position err (l_s)", "misfit as circle", "misfit as ellipse",
             "change"])

drop = np.array([1 - r.misfit / max(c.misfit, 1e-30)
                 for r, c in zip(res_tr, res_e)])
pos_tr = np.array([r.position_error_ls for r in res_tr])
mis_tr = np.array([r.misfit for r in res_tr])
print(f"\nmedian misfit reduction from releasing (b, alpha): {np.median(drop):+.1%}")
print(f"median position error, ellipse family: {np.median(pos_tr):.4f} lambda_s "
      f"(gate {cfg.GATE_POSITION_LS})")
print(f"improved in {int((drop > 0).sum())}/{len(drop)} cases")

# The graded version: the same numbers, per eccentricity tier, with axis 1 alongside.
table([(nm, f"{np.mean(th_e[m_, 2]/th_e[m_, 3]):.2f}", f"{np.median(fw_e[m_]):.4f}",
        f"{np.median(pos_tr[m_]):.4f}", f"{np.median(mis_tr[m_]):.3e}",
        f"{np.median(drop[m_]):+.0%}")
       for nm, m_ in [("mild", mild), ("wide", ~mild)]],
      ["tier", "mean a/b", "forward rel err", "position err (l_s)",
       "misfit as ellipse", "misfit change"])
print("prediction, then recovery, then what releasing (b, alpha) bought -- one row per "
      "tier,\nso a failure can be attributed to eccentricity or acquitted of it.")

In [ ]:
n_show = min(3, len(res_tr))
fig, ax = plt.subplots(1, n_show + 1, figsize=(3.0 * (n_show + 1), 3.1))
gx = np.arange(cfg.N_NET) * cfg.DX_NET
for k in range(n_show):
    a_ = ax[k]
    _, chi_true = geometry_channels(torch.as_tensor(th_e[k:k+1], device=DEV), ellipse)
    _, chi_c = geometry_channels(res_e[k].theta.unsqueeze(0).to(DEV), circle)
    _, chi_t = geometry_channels(res_tr[k].theta.unsqueeze(0).to(DEV), ellipse)
    a_.imshow(_np(chi_true)[0], origin="lower", cmap="Greys", vmin=0, vmax=1,
              extent=[gx[0], gx[-1], gx[0], gx[-1]])
    a_.contour(gx, gx, _np(chi_c)[0], levels=[0.5], colors="C3", linewidths=1.3)
    a_.contour(gx, gx, _np(chi_t)[0], levels=[0.5], colors="C2", linewidths=1.3)
    sx, sy = cfg.SOURCE_XY[int(src_e[k])]
    a_.plot(sx, sy, "C0*", ms=10)
    a_.set(title=f"ellipse {k}: grey truth,\nred circle fit, green ellipse fit",
           xticks=[], yticks=[])
    a_.grid(False)

a_ = ax[n_show]
a_.semilogy([0] * len(res_e), [r.misfit for r in res_e], "o", ms=5, alpha=0.7)
a_.semilogy([1] * len(res_tr), [r.misfit for r in res_tr], "s", ms=5, alpha=0.7)
for rc, rt in zip(res_e, res_tr):
    a_.semilogy([0, 1], [rc.misfit, rt.misfit], "-", c="0.6", lw=0.7)
a_.semilogy([-0.2, 0.2], [np.median(mis_e)] * 2, "k-", lw=1.6)
a_.semilogy([0.8, 1.2], [np.median([r.misfit for r in res_tr])] * 2, "k-", lw=1.6)
a_.axhline(np.median(mis_in), ls="--", c="C3", lw=1.0, label="in-family median")
a_.set(xticks=[0, 1], xticklabels=["circle\nfamily", "ellipse\nfamily"],
       ylabel="final misfit", title="no retraining, two extra\ndegrees of freedom")
a_.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "06_transfer_ellipse.png")
plt.show()

### Axis 2 -- the reference solver's opinion of the transfer

Forward prediction and inversion success are both measured against the surrogate's own idea of
the physics: the first compares it to FDTD at one geometry, the second only asks whether the
optimiser found a low value of a misfit the surrogate itself defines. Neither says the
surrogate's *sensitivities* on this shape point the right way, and sensitivities are what an
inversion actually consumes.

So the middle axis is stage E's step 9b asked again about an ellipse: central differences of the
receiver phasors with respect to all five parameters, once through the frozen network and once
through the reference solver, at the recovered geometry. That is `1 + 2 * 5` solves per step
size, spent on the *mildest* ellipse in the sample -- if the columns disagree there, they
disagree where the shape is closest to the training family, and eccentricity is not the
explanation available.

**Orientation is degenerate at low eccentricity, by construction.** Rotating a circle is the
identity, so `dJ/dalpha` vanishes as `a/b` approaches 1 and its finite-difference column becomes
small on both sides -- a ratio of two small numbers, which a worst-parameter gate would then
report as the whole story. The table prints all five columns and the verdict is read twice: over
every parameter, and over the four that are not degenerate. Which to believe is stated in
advance rather than picked afterwards -- the restricted one, for the reason just given.

`verify_with_solver` then re-solves the recovered ellipse, which is the review's actual request:
evaluate the final recovered geometry with the reference solver, not with the surrogate that
produced it. This data carries `SNR` dB of noise, so the truth's own residual should sit well
above the model-error floor and `J(theta_hat) / J(theta_true)` should be a measurement rather
than a division of roundoff by roundoff; `truth_at_floor` reports which of the two happened.

In [ ]:
k_m = int(np.argmin(np.where(mild, th_e[:, 2] / th_e[:, 3], np.inf)))
ce5_m = InverseCase(d_obs=cases_e[k_m].d_obs, src_idx=int(src_e[k_m]),
                    nu_idx=int(nu_e[k_m]), snr_db=SNR,
                    theta_true=torch.tensor(th_e[k_m], dtype=torch.float32)).to(DEV)
th_hat_m = res_tr[k_m].theta.detach().unsqueeze(0).to(DEV)

sens_e = SENS.surrogate_vs_solver_sensitivity(
    fwd, ce5_m, ellipse, incident=inc, theta=th_hat_m, h_rel=SENS_H_F, progress=tqdm)
ver_e = SENS.verify_with_solver(res_tr[k_m], ce5_m, incident=inc, family=ellipse,
                                forward=fwd)
print(f"case {k_m}: aspect {th_e[k_m, 2]/th_e[k_m, 3]:.2f}, the mildest in the sample; "
      f"{sens_e['n_solves'] + ver_e['n_solves']} FDTD solves at nt = {cfg.NT}")

In [ ]:
def _ver_status(v):
    return v.get("independent_status", "incomplete").upper()


ratio = ("at floor" if ver_e["residual_ratio"] is None
         else f"{ver_e['residual_ratio']:.3f}")
print(f"\nthe solver on the recovered ellipse: misfit {ver_e['misfit_solver']:.3e} "
      f"vs {ver_e['misfit_reported']:.3e} reported, J/J_truth {ratio}, "
      f"optimism {ver_e['surrogate_optimism']:.3f}")
print(f"position error {ver_e['position_error_ls']:.4f} lambda_s: "
      f"{_ver_status(ver_e)}  "
      f"(truth_at_floor {ver_e.get('truth_at_floor')})")

### The baseline on the same data

The RingCNN has three output numbers and no notion of an ellipse, so the most it can do is
report the equal-area circle. It is given exactly the data the inversion was given -- the same
phasors, normalised by the same receiver-space scale -- and scored by the same function:
`CNN.score` builds one `InversionResult` per sample and hands the list to `INV.summarise`, so
every number in the table below has the same definition on both sides. It did not until
recently; the baseline used to difference the first two columns of theta and the pipeline used
to permutation-match blobs and require an overlap gate, which made the two columns two
measurements rather than two estimators.

The mechanism the table is testing: the regressor learned a map from ring data to three numbers
on a distribution of circles, and an ellipse is off that distribution with no mechanism in the
network to notice. The inversion carries a forward model, so it can be handed a different family
and a different parameter count at inference time, and it reports a misfit that indicates when it
is out of its depth. Whether that translates into a smaller error is what the numbers say, not
something to assert in advance -- and on in-distribution circles the baseline is expected to be
competitive or better.

Two columns, and the second is the one that carries the claim. Position error is computed after
reducing the truth to its equal-area circle, so it is blind to shape by construction: a circle
placed at the centre of a 2.5:1 ellipse scores a *zero* position error. IoU compares each shape
through its own SDF and does not reduce anything, which is why an exactly-placed equal-area
circle still scores about 0.56 against that ellipse. A method that only ever returns circles has
a ceiling in the IoU column and none in the position column.

In [ ]:
def cnn_on(d, src_idx, nu_idx, theta_true, truth_family):
    s = torch.as_tensor(src_idx)
    j = torch.as_tensor(nu_idx)
    nu = torch.tensor([cfg.NU_LIST[int(k)] for k in j], dtype=torch.float32)
    x = CNN.pack_ring(d.cpu(), src_idx=s, nu=nu, scale=inc_scale_r[s, j])
    rd = CNN.RingData(x, torch.as_tensor(theta_true, dtype=torch.float32), nu, s)
    return CNN.score(net, rd.to(DEV), truth_family=truth_family)


cnn_e = cnn_on(d_e, src_e, nu_e, th_e, ellipse)
cnn_t = cnn_on(d_t, src_t, nu_t, th_t, twocircle)
inv_e = INV.summarise(res_e)
inv_t = INV.summarise(res_t)
inv_in = INV.summarise(res_in)

table([("circle test split", f"{sc_te['position_ls_median']:.4f}",
        f"{inv_in['position_ls_median']:.4f}",
        f"{sc_te['iou_median']:.3f}", f"{inv_in['iou_median']:.3f}"),
       ("ellipse", f"{cnn_e['position_ls_median']:.4f}",
        f"{inv_e['position_ls_median']:.4f}",
        f"{cnn_e['iou_median']:.3f}", f"{inv_e['iou_median']:.3f}"),
       ("two voids", f"{cnn_t['position_ls_median']:.4f}",
        f"{inv_t['position_ls_median']:.4f}",
        f"{cnn_t['iou_median']:.3f}", f"{inv_t['iou_median']:.3f}")],
      ["median over cases", "position, RingCNN", "position, inversion",
       "IoU, RingCNN", "IoU, inversion"])
print("position error in lambda_s, against the truth's equal-area circle; "
      "IoU against the true shape")
print(f"\nRingCNN: {net.n_params():,} parameters, one forward pass per case.")
print(f"Inversion: {int(np.median([r.n_forward for r in res_in]))} surrogate "
      f"evaluations, {np.median([r.seconds for r in res_in]):.1f} s per case.")
print("The baseline is cheap and it is not wrong -- it is just unable to say when it "
      "is.")

## Part 5 -- the thesis table

Every headline number, with the notebook that produced it and the gate it is measured against.
Read from the JSON records in `E.results`, so this cell reports what was actually run rather
than what was intended; anything missing shows as `-` instead of silently defaulting.

In [ ]:
def load_rec(name):
    p = E.results / name
    return json.loads(p.read_text()) if p.exists() else {}


r1 = load_rec("01_solver_validation.json")
r2 = load_rec("02_dataset_generation.json")
r3 = load_rec("03_train_fno.json")
r4 = load_rec("04_forward_eval.json")
r5 = load_rec("05_inversion.json")


def pick(d, *path, default=None):
    for k in path:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def fmt(v, spec=".4g"):
    return "-" if v is None else format(v, spec) if isinstance(v, float) else str(v)


ck = pick(r1, "checks", default={}) or {}


def check(substr, field="value"):
    for k, v in ck.items():
        if substr in k.lower():
            return v.get(field)
    return None


rows = [
    ("solver checks passed", f"{pick(r1,'n_pass',default='-')} of "
     f"{pick(r1,'n_total',default='-')}", "10 of 10", "01"),
    ("deconvolution amplification",
     fmt(pick(r1, "deconvolution", "worst_amplification")), "< 25", "01"),
    ("grid convergence, 256 vs 512", fmt(check("grid convergence")),
     f"< {cfg.GATE_GRID_CONVERGENCE}", "01"),
    ("absorber residual energy", fmt(check("absorber residual"), ".2e"),
     f"< {cfg.GATE_ABSORBER_RESIDUAL}", "01"),
    ("incident field vs Green's tensor", fmt(check("green"), ".2%"),
     f"< {cfg.GATE_GREEN_REL_L2:.0%}", "01"),
    ("labels vs traction-free cavity", fmt(check("cavity"), ".2%"),
     f"< {cfg.GATE_CAVITY_REL_L2:.0%}", "01"),
    ("cavity error under 3x grid refinement",
     fmt(check("interface width"), ".2%"), "< 2 points of shift", "01"),
    ("absorber vs open domain", fmt(check("open domain"), ".2%"),
     f"< {cfg.GATE_ABSORBER_REFLECTION:.0%}", "01"),
    ("physics residual on solver labels",
     fmt(check("physics residual"), ".3f"),
     f"< {cfg.GATE_PHYS_RESIDUAL_LABEL}", "01"),
    ("dataset re-solve spot check", fmt(pick(r2, "spot_check", "max"), ".2e"),
     f"< {pick(r2,'spot_check','gate',default='-')}", "02"),
    ("wrap-around tail energy",
     fmt(pick(r2, "tail_energy_fraction", "max"), ".2e"), "< 1e-3", "02"),
    ("surrogate field rel-L2 (test)", fmt(pick(r4, "test", "rel_l2")),
     f"< {cfg.GATE_REL_L2}", "04"),
    ("surrogate arrival error", fmt(pick(r4, "test", "phase")),
     f"< {cfg.GATE_ARRIVAL_PERIODS} periods", "04"),
    ("held-out-source penalty",
     fmt(pick(r4, "per_sample", "heldout_penalty")), "reported", "04"),
    ("physics-loss ablation, rel-L2",
     f"{fmt(pick(r3,'arms','nophys','rel_l2'))} -> "
     f"{fmt(pick(r3,'arms','full','rel_l2'))}", "physics <= none", "03"),
    ("L_phys floor (true field)", fmt(pick(r3, "phys_floor_true_field"), ".4e"),
     "reported", "03"),
    ("gradient check, significant figures",
     fmt(pick(r5, "gradient_check", "digits"), ".2f"),
     f">= {cfg.GATE_GRAD_SIGFIGS}", "05"),
    ("inversion success rate at 30 dB",
     fmt(pick(r5, "step11_30db", "all", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "05"),
    ("  on held-out illuminations",
     fmt(pick(r5, "step11_30db", "heldout", "success_rate"), ".1%"),
     f">= {cfg.GATE_SUCCESS_RATE:.0%}", "05"),
    ("basin width along/across",
     f"{fmt(pick(r5,'basin','full','along_ls'),'.3f')} / "
     f"{fmt(pick(r5,'basin','full','across_ls'),'.3f')} lambda_s",
     "~0.25, elongated", "05"),
    ("RingCNN baseline, median position",
     f"{sc_te['position_ls_median']:.4f} lambda_s", "for comparison", "06"),
    ("inversion, median position (in family)",
     f"{inv_in['position_ls_median']:.4f} lambda_s",
     f"< {cfg.GATE_POSITION_LS}", "06"),
    ("median IoU on the ellipse set, RingCNN / inversion",
     f"{cnn_e['iou_median']:.3f} / {inv_e['iou_median']:.3f}",
     f"> {cfg.GATE_IOU} to pass", "06"),
    ("lack-of-fit FPR (held out, frozen threshold)",
     f"{ev['fpr']:.2f} at TPR {ev['tpr']:.2f}", f"<= {cfg.GATE_LOF_FPR:.2f}", "06"),
    ("lack-of-fit AUC (diagnostic)", f"{roc['auc']:.4f}", "> 0.9 desirable", "06"),
    ("ellipse transfer, median position",
     f"{np.median(pos_tr):.4f} lambda_s", f"< {cfg.GATE_POSITION_LS}", "06"),
    ("ellipse transfer, misfit change",
     f"{np.median(drop):+.1%}", "negative is a failure", "06"),
]
table(rows, ["quantity", "value", "gate / expectation", "notebook"])

In [ ]:
    "transfer_sensitivity": dict(
        case=k_m, aspect_ratio=float(th_e[k_m, 2] / th_e[k_m, 3]),
        rel_worst_no_alpha=float(rel4), cosine_worst_no_alpha=float(cos4),
        gate_pass_no_alpha=bool(pass4),
        sensitivity={k: v for k, v in sens_e.items() if k not in ("rows", "theta")},
        solver_verify={k: v for k, v in ver_e.items()
                       if k not in ("theta", "theta_true", "family")},
        solver_verify_gates=dict(
            position_ls=cfg.GATE_SOLVER_VERIFY_LS,
            residual_ratio=cfg.GATE_SOLVER_VERIFY_RESIDUAL_RATIO,
            axis_ratio=cfg.GATE_SOLVER_VERIFY_AXIS_RATIO,
            orientation_deg=cfg.GATE_SOLVER_VERIFY_ORIENTATION_DEG)),

## The never-cut checklist

§11.3, in order. These are the things that stay in even when time runs out, because each one is
the only place a particular kind of silent wrongness can be caught. A `FAIL` or a `-` below is
not a note in the discussion section; it is a number that should not be quoted.

In [ ]:
checks = [
    ("solver sanity checks 1-5",
     (pick(r1, "n_pass") == pick(r1, "n_total") and pick(r1, "include_slow"))
     if r1 else None,
     "an unvalidated solver makes every downstream metric a report on the wrong "
     "physics"),
    ("dataset re-solve spot check", pick(r2, "spot_check", "passed"),
     "the file on disk is what the solver produced"),
    ("forward rel-L2 and arrival gates",
     all(pick(r4, "gates", default={}).values()) if pick(r4, "gates") else None,
     "the surrogate is the operator being inverted; its error is a floor"),
    ("held-out-source generalisation reported",
     pick(r4, "per_sample", "heldout_penalty") is not None,
     "the honest split -- SRC_HELDOUT appears in test and nowhere else"),
    ("gradient check to 3 significant figures",
     pick(r5, "gradient_check", "passed"),
     "a wrong gradient still converges, to the wrong answer"),
    ("misfit landscape figure",
     pick(r5, "basin", "full", "along_ls") is not None,
     "the basin width and its elongation are the resolution claim"),
    ("success rate at 30 dB",
     pick(r5, "step11_30db", "all", "gate_pass"),
     "the headline inverse result"),
    ("SNR sweep", bool(pick(r5, "snr_sweep")),
     "says whether 30 dB is inside the working range or on a cliff edge"),
    ("out-of-family transfer, no retraining",
     bool(np.median(drop) > 0), "the §8.5 claim, and the reason for a forward model"),
    ("graded: the mild tier is measured on its own",
     bool(np.median(pos_tr[mild]) <= cfg.GATE_POSITION_LS),
     "a single hard tier cannot separate 'eccentricity broke it' from 'the transfer "
     "broke it'; the mild tier is the one eccentricity cannot be blamed for"),
    ("out-of-family sensitivities checked against the solver",
     bool(pass4),
     "the transfer claim is about sensitivities, not just about one predicted field, "
     "and only the reference solver can adjudicate them off the training family"),
    ("solver position error on transferred ellipse (step 10b)",
     ver_e.get("position_gate_pass", False),
     "the recovered geometry re-solved with a model the optimiser could not exploit; "
     "uses separate position, residual, and shape criteria"),
    ("lack-of-fit FPR at the frozen threshold",
     bool(ev["gate_pass"]),
     "a confident wrong answer on an unmodelled defect is the worst failure mode, and a "
     "rate is only a rate if the threshold was frozen before the test half"),
    ("CNN baseline for comparison", bool(sc_te), "otherwise there is no claim"),
]

## What is left out, on purpose

- **No mixed precision anywhere.** The spectral weights are complex and the gradient check of
  notebook 05 needs double precision; TF32 is off for the same reason. The speedup was not
  worth an unexplained loss of three digits.
- **The screen is a position scan, in every family.** `screen_candidates` reads `param_names`,
  so it moves the centres of an ellipse or of a two-void pair too -- but it leaves every
  non-centre parameter at the midpoint of its bounds, because scanning all five would be a
  `16^5` lattice. So an out-of-family inversion is seeded from the converged circle fit instead
  of screened over its own shape parameters. That is a real limitation, and it is why the
  transfer experiment passes `skip_screen=True`.
- **`Ellipse` and `TwoCircle` are never trained on.** They exist to be transferred to. Adding
  them to the training distribution would make the §8.5 result vacuous.
- **The equal-area circle is a convention.** There is no correct circle for an ellipse, which is
  why the lack-of-fit indicator is built on the residual and not on the position error.
- **The indicator does not identify shapes.** It answers "could a circle have made this data?".
  A flagged case is not thereby an ellipse: the same residual is consistent with a wrong source,
  a wrong material, or an optimiser that stopped early.
- **The out-of-family sensitivity check is one case, one illumination, one step size.** It costs
  `1 + 2 * 5` full FDTD solves for a single mild ellipse; a rate over the tier would be that
  many times six. It says whether the columns agree where the shape is closest to the training
  family -- the attribution the graded tiers exist for -- not that they agree everywhere.